# MS COCO Hybrid KD: three-seed audit

Notebook CPU này kiểm tra độc lập ba seed `42`, `123`, `2026`, xác minh
manifest/checkpoint, tổng hợp chỉ số theo thang ×100 và đối chiếu mô tả với
Sequence KD cùng Dense Mapped Logit-KD. Notebook chỉ đọc Validation; Held-out
Test vẫn khóa.

## Input bắt buộc

1. Output `HYB_42`;
2. output `HYB_123`;
3. output `HYB_2026`;
4. output aggregate Sequence KD có `teacher_a_s1_sequence_kd_three_seed_summary.json`;
5. output aggregate Dense Mapped có `xlkd06_three_seed_summary.json`.

Runtime: **CPU / Accelerator None / Internet Off**.

In [1]:
import csv
import hashlib
import json
import math
import os
import statistics
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "HYB_Audit"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 123, 2026)
METRICS = ("BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D")

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temp, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    if len(matches) > 1:
        preferred = [
            path for path in matches
            if "output_v2" in {part.lower() for part in path.parts}
        ]
        if len(preferred) == 1:
            return preferred[0]
    assert len(matches) == 1, (name, [str(path) for path in matches])
    return matches[0]

def verify_manifest(marker_path, marker):
    manifest_path = marker_path.parent / marker["manifest"]["csv"]["relative_path"]
    assert manifest_path.is_file(), manifest_path
    assert sha256(manifest_path) == marker["manifest"]["csv"]["sha256"]
    with manifest_path.open("r", encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(marker["manifest"]["csv"]["rows"])
    for row in rows:
        artifact = marker_path.parent / row["relative_path"]
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"]), artifact
        assert sha256(artifact) == row["sha256"], artifact
    return len(rows)

runs = {}
rows = []
for seed in SEEDS:
    marker_path = exactly_one(f"HYB_{seed}_COMPLETE.json")
    marker = read_json(marker_path)
    assert marker["status"] == "complete"
    assert marker["experiment_id"] == "EXPANSIONNET_S1_HYBRID_KD"
    assert marker["method"] == "Hybrid Sequence + Dense Mapped Logit-KD"
    assert int(marker["seed"]) == seed
    assert math.isclose(float(marker["temperature"]), 4.0)
    assert math.isclose(float(marker["alpha_human_ce"]), 0.50)
    assert math.isclose(float(marker["alpha_sequence_ce"]), 0.25)
    assert math.isclose(float(marker["alpha_kd"]), 0.25)
    assert marker["production_contract"]["status"] == "locked"
    assert marker["manifest_verification"] == "all_passed"
    assert marker["selection_split"] == "validation"
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
    manifest_rows = verify_manifest(marker_path, marker)

    for checkpoint_name in ("best_checkpoint", "last_checkpoint"):
        checkpoint = marker[checkpoint_name]
        checkpoint_path = marker_path.parent / checkpoint["relative_path"]
        assert checkpoint_path.is_file()
        assert sha256(checkpoint_path) == checkpoint["sha256"]

    comparison_path = marker_path.parent / "results" / f"HYB_{seed}_vs_S1.json"
    comparison = read_json(comparison_path)
    best = marker["best_result"]
    hybrid = comparison[f"hybrid_kd_seed{seed}"]
    baseline = comparison[f"baseline_seed{seed}"]
    delta = comparison["delta_kd_minus_baseline"]
    assert best["prediction_count"] == 5000
    assert best["test_set_used"] is False
    for metric in METRICS:
        assert math.isclose(float(best["metrics"][metric]), float(hybrid[metric]), abs_tol=1e-12)
        assert math.isclose(float(hybrid[metric] - baseline[metric]), float(delta[metric]), abs_tol=1e-12)
    assert math.isclose(float(best["teacher_forced_ce"]), float(hybrid["CE"]), abs_tol=1e-12)

    runs[seed] = {"path": marker_path, "marker": marker, "comparison": comparison}
    rows.append({
        "seed": seed,
        "completed_epochs": int(marker["completed_epochs"]),
        "best_epoch": int(best["epoch"]),
        "optimizer_steps_at_best": int(best["global_optimizer_step"]),
        "BLEU_1_x100": round(float(hybrid["BLEU-1"]) * 100, 2),
        "BLEU_4_x100": round(float(hybrid["BLEU-4"]) * 100, 2),
        "METEOR_x100": round(float(hybrid["METEOR"]) * 100, 2),
        "ROUGE_L_x100": round(float(hybrid["ROUGE-L"]) * 100, 2),
        "CIDEr_D_x100": round(float(hybrid["CIDEr-D"]) * 100, 2),
        "CE_raw": round(float(hybrid["CE"]), 6),
        "delta_CIDEr_D_x100": round(float(delta["CIDEr-D"]) * 100, 2),
        "manifest_rows": manifest_rows,
    })

summary = {}
for metric in METRICS + ("CE",):
    hybrid_values = [
        float(runs[seed]["comparison"][f"hybrid_kd_seed{seed}"][metric])
        for seed in SEEDS
    ]
    baseline_values = [
        float(runs[seed]["comparison"][f"baseline_seed{seed}"][metric])
        for seed in SEEDS
    ]
    delta_values = [
        float(runs[seed]["comparison"]["delta_kd_minus_baseline"][metric])
        for seed in SEEDS
    ]
    summary[metric] = {
        "hybrid_mean": statistics.mean(hybrid_values),
        "hybrid_sample_sd": statistics.stdev(hybrid_values),
        "baseline_mean": statistics.mean(baseline_values),
        "baseline_sample_sd": statistics.stdev(baseline_values),
        "paired_delta_mean": statistics.mean(delta_values),
        "paired_delta_sample_sd": statistics.stdev(delta_values),
    }

sequence_summary_path = exactly_one("teacher_a_s1_sequence_kd_three_seed_summary.json")
dense_summary_path = exactly_one("xlkd06_three_seed_summary.json")
sequence_summary = read_json(sequence_summary_path)
dense_summary = read_json(dense_summary_path)
sequence_cider = float(sequence_summary["aggregate"]["CIDEr-D"]["kd"]["mean"])
dense_cider = float(dense_summary["aggregate"]["CIDEr-D"]["kd"]["mean"])
hybrid_cider = float(summary["CIDEr-D"]["hybrid_mean"])

decision = (
    "AGGREGATE_PASSED_COMPARISON_REQUIRED"
    if all(
        runs[seed]["comparison"]["delta_kd_minus_baseline"]["CIDEr-D"] > 0
        for seed in SEEDS
    )
    else "STOP_NO_CONSISTENT_BASELINE_GAIN"
)

csv_path = OUTPUT_ROOT / "HYB_METRICS_X100.csv"
with csv_path.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

payload = {
    "schema_version": "hyb_audit_v1",
    "created_at_utc": now(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "method": "Hybrid Sequence and Dense-Mapped Logit Knowledge Distillation",
    "seeds": list(SEEDS),
    "protocol": {
        "temperature": 4.0,
        "alpha_human_ce": 0.50,
        "alpha_sequence_ce": 0.25,
        "alpha_kd": 0.25,
        "selection_split": "validation",
    },
    "summary": summary,
    "descriptive_comparison": {
        "hybrid_CIDEr_D_x100": round(hybrid_cider * 100, 2),
        "sequence_KD_CIDEr_D_x100": round(sequence_cider * 100, 2),
        "dense_mapped_CIDEr_D_x100": round(dense_cider * 100, 2),
        "hybrid_minus_sequence_x100": round((hybrid_cider - sequence_cider) * 100, 2),
        "hybrid_minus_dense_mapped_x100": round((hybrid_cider - dense_cider) * 100, 2),
    },
    "gates": {
        "three_seed_outputs_complete": True,
        "manifest_and_checkpoint_integrity": True,
        "protocol_consistency": True,
        "positive_CIDEr_D_delta_vs_baseline": "3 / 3",
        "paired_method_comparison_complete": False,
        "final_method_gate": "pending",
    },
    "decision": decision,
    "next": "Run HYB_Compare on Validation only.",
    "fairness_disclosure": (
        "Hybrid KD is a post-baseline continuation. Descriptive gains over the original "
        "baseline do not by themselves establish a matched-budget causal gain."
    ),
    "held_out_test_may_be_opened": False,
    "onnx_or_int8_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "sources": {
        "hybrid_markers": {
            str(seed): {"path": str(runs[seed]["path"]), "sha256": sha256(runs[seed]["path"])}
            for seed in SEEDS
        },
        "sequence_summary_sha256": sha256(sequence_summary_path),
        "dense_summary_sha256": sha256(dense_summary_path),
    },
}
result_path = OUTPUT_ROOT / "HYB_AUDIT.json"
write_json(result_path, payload)

report = OUTPUT_ROOT / "REPORT.md"
report.write_text(
    "# MS COCO 2014 — Hybrid KD three-seed audit\n\n"
    f"- Trạng thái: **PASSED**; toàn vẹn manifest/checkpoint: **PASSED**.\n"
    f"- CIDEr-D Hybrid: **{hybrid_cider * 100:.2f} ± {summary['CIDEr-D']['hybrid_sample_sd'] * 100:.2f}**.\n"
    f"- Δ CIDEr-D so với baseline: **{summary['CIDEr-D']['paired_delta_mean'] * 100:+.2f}**; thắng **3/3 seed**.\n"
    f"- So với Sequence KD: **{(hybrid_cider - sequence_cider) * 100:+.2f}** điểm CIDEr-D.\n"
    f"- So với Dense Mapped Logit-KD: **{(hybrid_cider - dense_cider) * 100:+.2f}** điểm CIDEr-D.\n"
    "- Kết luận hiện thời: đạt cổng tái lập/tổng hợp, chưa đạt cổng chọn phương pháp cuối.\n"
    "- Held-out Test vẫn khóa; chưa cho phép ONNX/INT8.\n",
    encoding="utf-8",
)

artifacts = [result_path, csv_path, report]
write_json(OUTPUT_ROOT / "MANIFEST.json", {
    "schema_version": "hyb_audit_manifest_v1",
    "status": "complete",
    "artifacts": [
        {"relative_path": p.name, "size_bytes": p.stat().st_size, "sha256": sha256(p)}
        for p in artifacts
    ],
    "test_set_used": False,
})

print("Audit status              : PASSED")
print("Three seeds               : 3 / 3 COMPLETE")
print("Integrity                 : PASSED")
print("Hybrid CIDEr-D x100       :", round(hybrid_cider * 100, 2))
print("Delta vs baseline x100    :", round(summary["CIDEr-D"]["paired_delta_mean"] * 100, 2))
print("Hybrid - Sequence x100    :", round((hybrid_cider - sequence_cider) * 100, 2))
print("Hybrid - DenseMap x100    :", round((hybrid_cider - dense_cider) * 100, 2))
print("Decision                  :", decision)
print("Held-out Test             : SEALED")

Audit status              : PASSED
Three seeds               : 3 / 3 COMPLETE
Integrity                 : PASSED
Hybrid CIDEr-D x100       : 109.27
Delta vs baseline x100    : 10.69
Hybrid - Sequence x100    : -1.37
Hybrid - DenseMap x100    : 1.21
Decision                  : AGGREGATE_PASSED_COMPARISON_REQUIRED
Held-out Test             : SEALED


## Cổng kế tiếp

Chỉ khi audit trả về `AGGREGATE_PASSED_COMPARISON_REQUIRED` mới chạy
`HYB_Compare`. Không mở Test và không xuất ONNX/INT8 ở bước này.